In [1]:
!uv sync

Resolved 17 packages in 22ms
Audited 16 packages in 4ms


In [ ]:
# Imports
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from mandrop.engine import make_step, compute_macros, init_state
from mandrop.generator import setup, boundary_stats
from mandrop.run import run
from mandrop.viz import plot_fields

print(f"JAX {jax.__version__}, devices: {jax.devices()}")

In [ ]:
# Simulation parameters
Nx, Ny = 200, 400       # domain size (lattice units)

W = 3.0                 # interface width
sigma = 0.01            # surface tension
beta  = 3.0 * sigma / W
kappa = 6.0 * sigma * W

rho0 = 1.0              # base density
nu   = 1.0 / 6.0        # kinematic viscosity
tau_f = 3.0 * nu + 0.5  # relaxation time
M_ch = 0.01             # Cahn-Hilliard mobility
drho = 0.001             # pressure drop (inlet-outlet)

In [ ]:
# Geometry and step function
geo = setup(
    Nx=Nx, Ny=Ny,
    rho_in_water=rho0 + drho / 2.0,
    rho_in_oil=rho0 + drho / 2.0,
    rho_out=rho0 - drho / 2.0,
)
p = geo["params"]

step = make_step(
    geo["wall"], geo["fluid"], geo["interior"], geo["opp_jnp"],
    tau_f, beta, kappa, M_ch,
    geo["apply_f_bcs"], geo["apply_phi_bcs"], geo["boundary_mask"],
)

print(f"Domain: {Nx}×{Ny}, channel={p['w_channel']}, side={p['w_side']}")
print(f"Junction y={p['junction_y']} [{p['jy_bot']},{p['jy_top']}]")

In [ ]:
# Initial state: equilibrium f + water prefill in upper chip region
f0, phi0 = init_state(Nx, Ny, rho0, geo["apply_phi_bcs"], geo["water_prefill"])
interior = geo["interior"]

print(f"Water prefill: {int(geo['water_prefill'].sum())} pixels")
print(f"Initial water (φ<0.5): {int((phi0 < 0.5).sum())} pixels")

In [ ]:
# Visualize geometry and initial phi distribution
fig, axes = plt.subplots(1, 2, figsize=(8, 12))

im0 = axes[0].imshow(geo["wall"].T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
axes[0].set_title('Wall mask')
axes[0].axhline(p['jy_top'], color='r', ls=':', alpha=0.5, label='junction top')
axes[0].axhline(p['jy_bot'], color='b', ls=':', alpha=0.5, label='junction bot')
axes[0].legend(loc='upper right')
plt.colorbar(im0, ax=axes[0], shrink=0.5)

im1 = axes[1].imshow(phi0.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
axes[1].set_title('Initial φ (oil=1, water=0)')
plt.colorbar(im1, ax=axes[1], shrink=0.5)

for ax in axes:
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Run simulation
f_final, phi_final, total_steps = run(
    step, f0, phi0, interior, p,
    chunk_size=500, n_chunks=60,
)

In [ ]:
# Summary statistics
rho_final, ux_final, uy_final = compute_macros(f_final)
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)
n_water = ((phi_final < 0.5).astype(jnp.float64) * interior).sum()

print(f"FLOW-FOCUSING — {n_chunks*chunk_size} STEPS")
print(f"Max velocity: {vel_mag.max():.4e} lu")
print(f"Water pixels: {n_water:.0f}")
print(f"phi range:    [{phi_final.min():.6f}, {phi_final.max():.6f}]")
print(f"rho range:    [{rho_final.min():.6f}, {rho_final.max():.6f}]")

In [ ]:
# Final field plots: φ, ρ, u_y, |u|
plot_fields(f_final, phi_final)